# PaperRAG Qasper Eval Notebook

这个 notebook 是唯一推荐的评测入口。

用法就是：上传这一个 notebook 到 Colab，然后从上到下依次运行。

它会完成这些事情：

1. 克隆 `PaperRAG` 仓库
2. 安装 benchmark 依赖
3. 直接在 notebook 里覆盖环境参数，不依赖上传 `.env`
4. 运行 Qasper benchmark
5. 输出和当前项目一致格式的 `evalout/` 目录


## Notebook Config

直接修改下面的配置 cell 即可。

- 公开仓库不需要 GitHub token
- Milvus 和模型参数已经直接写在 notebook 里
- 跑完后把有效参数回填到本地 `.env`


In [1]:
from pathlib import Path

# ===== repo =====
REPO_OWNER = 'YaoHui-Wu06022'
REPO_NAME = 'PaperRAG'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/PaperRAG')

# ===== run target =====
DATASET = 'qasper'
LIMIT = 50
SEED = 88
SHUFFLE = True
WITH_LLM = False
WITH_RAGAS = False

# ===== output =====
OUTPUT_DIR = Path('/content/evalout')

# ===== vector db =====
# MILVUS_MODE: 'cloud' or 'lite'
MILVUS_MODE = 'lite'
MILVUS_URI = 'milvus_lite'
MILVUS_TOKEN = ''
MILVUS_DB_NAME = ''
MILVUS_LITE_URI = '/content/milvus_lite/qasper_eval.db'

# ===== parameter overrides =====
# 你主要就在这里改参数，效果满意后再回本地改 .env
ENV_OVERRIDES = {
    'VECTOR_BACKEND': 'milvus',
    'EMBEDDING_PROVIDER': 'huggingface',
    'EMBEDDING_MODEL': 'BAAI/bge-m3',
    'RETRIEVAL_MODE': 'hybrid',
    'RETRIEVER_TOP_K': '30',
    'FINAL_TOP_K': '5',
    'HYBRID_DENSE_TOP_K': '30',
    'HYBRID_BM25_TOP_K': '30',
    'HYBRID_RRF_K': '60',
    'QUERY_REWRITE_ENABLED': 'true',
    'QUERY_REWRITE_MAX_VARIANTS': '3',
    'USE_RERANKER': 'true',
    'RERANK_CONDITIONAL_ENABLED': 'false',
    'DIVERSIFY_BY_SOURCE': 'true',
    'MAX_CHUNKS_PER_SOURCE': '5',
    'RETRIEVAL_SCORE_THRESHOLD_MODE': 'absolute',
    'RETRIEVAL_SCORE_THRESHOLD': '0.05',
    'RETRIEVAL_SCORE_RELATIVE_RATIO': '0.7',
    'CHUNK_SIZE': '512',
    'CHUNK_OVERLAP': '128',
    'CHUNK_MIN_BLOCK_CHARS': '180',
}

CONFIG_KEYS_TO_SHOW = [
    'vector_backend', 'embedding_model', 'retrieval_mode', 'retriever_top_k', 'final_top_k',
    'hybrid_dense_top_k', 'hybrid_bm25_top_k', 'hybrid_rrf_k', 'query_rewrite_enabled',
    'query_rewrite_max_variants', 'use_reranker', 'rerank_conditional_enabled',
    'diversify_by_source', 'max_chunks_per_source', 'retrieval_score_threshold',
    'retrieval_score_threshold_mode', 'retrieval_score_relative_ratio',
    'chunk_size', 'chunk_overlap', 'chunk_min_block_chars'
]

print({
    'repo': f'{REPO_OWNER}/{REPO_NAME}@{REPO_BRANCH}',
    'dataset': DATASET,
    'limit': LIMIT,
    'shuffle': SHUFFLE,
    'output_dir': str(OUTPUT_DIR),
    'milvus_mode': MILVUS_MODE,
    'with_llm': WITH_LLM,
    'with_ragas': WITH_RAGAS,
})


{'repo': 'YaoHui-Wu06022/PaperRAG@main', 'dataset': 'qasper', 'limit': 50, 'shuffle': True, 'output_dir': '/content/evalout', 'milvus_mode': 'lite', 'with_llm': False, 'with_ragas': False}


In [8]:
import torch

print('cuda_available=', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu=', torch.cuda.get_device_name(0))
else:
    print('warning: current runtime has no GPU, switch Colab runtime to GPU first')


cuda_available= True
gpu= Tesla T4


In [2]:
import os
import shutil
import subprocess

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repo_ready=', REPO_DIR)


repo_ready= /content/PaperRAG


In [3]:
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'python-dotenv>=1.0.1',
        'pymilvus>=2.5.0',
        'milvus-lite>=2.4.0',
        'langchain-milvus>=0.1.7',
        'langchain>=0.3.7',
        'langchain-core>=0.3.15',
        'langchain-community>=0.3.7',
        'langchain-text-splitters>=0.3.2',
        'langchain-huggingface>=0.1.0',
        'langchain-openai>=0.2.5',
        'sentence-transformers>=3.2.1',
        'transformers>=4.41,<5',
        'openai>=1.54.0',
        'requests>=2.32.3',
        'pandas>=2.2.3',
        'datasets>=3.1.0',
        'rank-bm25>=0.2.2',
    ], check=True)

if WITH_RAGAS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ragas>=0.2.3'], check=True)

print('dependencies_ready')


dependencies_ready


In [2]:
import json
import os
import torch

os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_EMBED_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['HF_EMBED_BATCH_SIZE'] = '64' if torch.cuda.is_available() else '16'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

milvus_mode = str(MILVUS_MODE).strip().lower()
active_milvus_uri = str(MILVUS_URI)
os.environ['MILVUS_MODE'] = milvus_mode
if milvus_mode == 'lite':
    active_milvus_uri = str(MILVUS_LITE_URI)
    os.environ.pop('MILVUS_URI', None)
    os.environ['MILVUS_LITE_URI'] = active_milvus_uri
    os.environ['MILVUS_TOKEN'] = ''
    os.environ['MILVUS_DB_NAME'] = ''
else:
    os.environ['MILVUS_URI'] = active_milvus_uri
    os.environ.pop('MILVUS_LITE_URI', None)
    os.environ['MILVUS_TOKEN'] = str(MILVUS_TOKEN)
    os.environ['MILVUS_DB_NAME'] = str(MILVUS_DB_NAME)
for key in ('AIHUBMIX_API_KEY', 'AIHUBMIX_BASE_URL', 'OPENAI_API_KEY', 'OPENAI_BASE_URL'):
    os.environ.pop(key, None)

for key, value in ENV_OVERRIDES.items():
    os.environ[str(key)] = str(value)

print(json.dumps({
    'MILVUS_MODE': milvus_mode,
    'MILVUS_URI': active_milvus_uri,
    'MILVUS_URI_set': bool(active_milvus_uri),
    'MILVUS_DB_NAME': os.environ.get('MILVUS_DB_NAME', ''),
    'HF_EMBED_DEVICE': os.environ.get('HF_EMBED_DEVICE'),
    'HF_EMBED_BATCH_SIZE': os.environ.get('HF_EMBED_BATCH_SIZE'),
    'ENV_OVERRIDES': ENV_OVERRIDES,
}, ensure_ascii=False, indent=2))


{
  "MILVUS_MODE": "lite",
  "MILVUS_URI": "/content/milvus_lite/qasper_eval.db",
  "MILVUS_URI_set": true,
  "MILVUS_DB_NAME": "",
  "HF_EMBED_DEVICE": "cuda",
  "HF_EMBED_BATCH_SIZE": "64",
  "ENV_OVERRIDES": {
    "VECTOR_BACKEND": "milvus",
    "EMBEDDING_PROVIDER": "huggingface",
    "EMBEDDING_MODEL": "BAAI/bge-m3",
    "RETRIEVAL_MODE": "bm25",
    "RETRIEVER_TOP_K": "30",
    "FINAL_TOP_K": "5",
    "HYBRID_DENSE_TOP_K": "30",
    "HYBRID_BM25_TOP_K": "30",
    "HYBRID_RRF_K": "60",
    "QUERY_REWRITE_ENABLED": "true",
    "QUERY_REWRITE_MAX_VARIANTS": "3",
    "USE_RERANKER": "false",
    "RERANK_CONDITIONAL_ENABLED": "false",
    "DIVERSIFY_BY_SOURCE": "true",
    "MAX_CHUNKS_PER_SOURCE": "5",
    "RETRIEVAL_SCORE_THRESHOLD_MODE": "absolute",
    "RETRIEVAL_SCORE_THRESHOLD": "0.05",
    "RETRIEVAL_SCORE_RELATIVE_RATIO": "0.7",
    "CHUNK_SIZE": "512",
    "CHUNK_OVERLAP": "128",
    "CHUNK_MIN_BLOCK_CHARS": "180"
  }
}


In [3]:
import json
import os
import time
from pathlib import Path

def log_stage(message: str) -> None:
    now = time.strftime('%H:%M:%S')
    print(f'[{now}] {message}', flush=True)

benchmark_start = time.perf_counter()
os.chdir(REPO_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log_stage(f'Workspace ready: repo={REPO_DIR} output={OUTPUT_DIR}')

log_stage('Importing benchmark modules...')
from config import load_config
from colab_eval.dataset_benchmark import run_dataset_benchmark, save_dataset_benchmark_result

log_stage('Loading config from current notebook environment...')
config = load_config()
active_config = {key: getattr(config, key) for key in CONFIG_KEYS_TO_SHOW}
print(json.dumps({
    'dataset': DATASET,
    'limit': LIMIT,
    'shuffle': SHUFFLE,
    'with_llm': WITH_LLM,
    'with_ragas': WITH_RAGAS,
    'vector_backend': config.vector_backend,
    'embedding_model': config.embedding_model,
    'retrieval_mode': config.retrieval_mode,
    'retriever_top_k': config.retriever_top_k,
    'final_top_k': config.final_top_k,
}, ensure_ascii=False, indent=2), flush=True)

if config.vector_backend.strip().lower() == 'milvus':
    if str(MILVUS_MODE).strip().lower() == 'lite':
        log_stage('Milvus Lite benchmark will use a local db file and temporary collections in this Colab runtime.')
    else:
        log_stage('Milvus benchmark will build a temporary collection and drop it after evaluation.')

log_stage('Starting dataset benchmark...')
result = run_dataset_benchmark(
    config,
    dataset_key=DATASET,
    limit=LIMIT,
    seed=SEED,
    shuffle=SHUFFLE,
    with_llm=WITH_LLM,
    with_ragas=WITH_RAGAS,
)

log_stage('Saving benchmark artifacts...')
save_dataset_benchmark_result(result, OUTPUT_DIR)
(OUTPUT_DIR / 'effective_config.json').write_text(
    json.dumps(active_config, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

total_seconds = time.perf_counter() - benchmark_start
log_stage(f'Benchmark finished in {total_seconds:.1f}s')
print('saved_to=', OUTPUT_DIR)
print(json.dumps(result.summary, ensure_ascii=False, indent=2))


[12:24:09] Workspace ready: repo=/content/PaperRAG output=/content/evalout
[12:24:09] Importing benchmark modules...
[12:24:20] Loading config from current notebook environment...
{
  "dataset": "qasper",
  "limit": 50,
  "shuffle": true,
  "with_llm": false,
  "with_ragas": false,
  "vector_backend": "milvus",
  "embedding_model": "BAAI/bge-m3",
  "retrieval_mode": "bm25",
  "retriever_top_k": 30,
  "final_top_k": 5
}
[12:24:20] Milvus Lite benchmark will use a local db file and temporary collections in this Colab runtime.
[12:24:20] Starting dataset benchmark...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Preparing corpus:   0%|          | 0/50 [00:00<?, ?qa/s]

Evaluating:   0%|          | 0/50 [00:00<?, ?qa/s]

[12:25:35] Saving benchmark artifacts...
[12:25:35] Benchmark finished in 86.7s
saved_to= /content/evalout
{
  "dataset": "qasper",
  "samples": 50,
  "corpus_docs": 664,
  "corpus_raw_docs": 45,
  "corpus_chunks": 664,
  "corpus_parent_docs": 45,
  "vector_backend": "milvus",
  "vector_collection": "rag_bench_qasper_e662fb3d",
  "chunk_size": 512,
  "chunk_overlap": 128,
  "chunk_tokenizer_model": "BAAI/bge-m3",
  "chunk_use_structure_split": true,
  "chunk_min_block_chars": 180,
  "retriever_top_k": 30,
  "final_top_k": 5,
  "retrieval_mode": "bm25",
  "hybrid_dense_top_k": 30,
  "hybrid_bm25_top_k": 30,
  "hybrid_rrf_k": 60,
  "query_rewrite_enabled": true,
  "query_rewrite_max_variants": 3,
  "use_reranker": false,
  "rerank_conditional_enabled": false,
  "rerank_skip_min_top1_score": 0.55,
  "rerank_skip_min_score_gap": 0.08,
  "rerank_skip_min_rel_gap": 0.2,
  "diversify_by_source": true,
  "max_chunks_per_source": 5,
  "retrieval_score_threshold": 0.05,
  "retrieval_score_thresh

In [4]:
import json
import pandas as pd

summary = json.loads((OUTPUT_DIR / 'dataset_benchmark_summary.json').read_text(encoding='utf-8'))
detail = pd.read_csv(OUTPUT_DIR / 'dataset_benchmark_detail.csv')
effective_config = json.loads((OUTPUT_DIR / 'effective_config.json').read_text(encoding='utf-8'))

summary


{'dataset': 'qasper',
 'samples': 50,
 'corpus_docs': 664,
 'corpus_raw_docs': 45,
 'corpus_chunks': 664,
 'corpus_parent_docs': 45,
 'vector_backend': 'milvus',
 'vector_collection': 'rag_bench_qasper_e662fb3d',
 'chunk_size': 512,
 'chunk_overlap': 128,
 'chunk_tokenizer_model': 'BAAI/bge-m3',
 'chunk_use_structure_split': True,
 'chunk_min_block_chars': 180,
 'retriever_top_k': 30,
 'final_top_k': 5,
 'retrieval_mode': 'bm25',
 'hybrid_dense_top_k': 30,
 'hybrid_bm25_top_k': 30,
 'hybrid_rrf_k': 60,
 'query_rewrite_enabled': True,
 'query_rewrite_max_variants': 3,
 'use_reranker': False,
 'rerank_conditional_enabled': False,
 'rerank_skip_min_top1_score': 0.55,
 'rerank_skip_min_score_gap': 0.08,
 'rerank_skip_min_rel_gap': 0.2,
 'diversify_by_source': True,
 'max_chunks_per_source': 5,
 'retrieval_score_threshold': 0.05,
 'retrieval_score_threshold_mode': 'absolute',
 'retrieval_score_relative_ratio': 0.7,
 'retrieval_score_quantile': 0.6,
 'reranker_score_threshold': 0.0005,
 'ret